# Site suitability pipeline, step by step (Google Colab)

GRG 327G / CS 378 group project: Gunner Johnson, Jay Patil, Palash Pawar.

**AI note.** This notebook was written with Claude (Anthropic). It was checked for syntax only. **It has not been run**, because the data servers and Earth Engine could not be reached where it was written. Expect small fixes the first time through.

**How to use it.** Run the cells from top to bottom. Each step says what it needs from you and what to check before moving on. Cells marked **YOU** need a sign-in, an upload, or a decision.

**Rules this notebook follows**
- Every feature describes 2020 or earlier. The label is a new-construction permit in 2021–2025.
- Data stays in your Google Drive folder. Nothing here commits data to GitHub.
- No step invents data. If a file is missing, the step stops and tells you.

**It expects these scripts in the repo:** `pull_permits.py`, `pull_parcels.py`, `build_labels.py`, `ee_extract_embeddings.py` (with `--terrain-out`), `build_features.py`, `train_models.py`. Merge the pull requests that add `build_features.py` and `--terrain-out` before running steps 6 and 8.

## 0. Settings (YOU: edit this cell)

In [ ]:
# ---- Edit these -------------------------------------------------------------
GCP_PROJECT = ""                 # your Google Cloud project ID registered for Earth Engine
DATA_DIR = "/content/drive/MyDrive/geoai_site_suitability_data"   # data lives here, in your Drive
REPO = "GunnerJ5/geoai-site-suitability"

# Filled in as you learn them (steps 2, 4 and 5 tell you how):
PARCEL_ID_FIELD = ""             # id column in the 2020 parcel file that matches permits' tcad_id
PARCELS_2020 = "parcels_2020.geojson"   # file name inside DATA_DIR
PARCEL_YEAR = None               # the year that file describes, e.g. 2020 or 2019
APPRAISAL_CSV = "appraisal_2020.csv"    # id, land value, improvement value
APPRAISAL_YEAR = None            # e.g. 2020
APPRAISAL_ID_COL = ""            # id column in the appraisal CSV
LABELS_JOIN_COL = ""             # column in the parcel file holding the same id
LAND_VALUE_COL = ""
IMPROVEMENT_VALUE_COL = ""
MAX_IMPROVEMENT_RATIO = 0.25     # "vacant or underused" threshold. A TEAM DECISION: choose and report it.
# -----------------------------------------------------------------------------
CUTOFF_YEAR = 2020

## 1. Connect Google Drive and get the code

Data is written to a folder in your Drive so it survives when Colab disconnects.

The repository is private, so pick **one** way to get the code:
- **A (simplest):** on GitHub click *Code → Download ZIP*, then upload the zip with the second cell.
- **B:** clone with a GitHub token. Create a fine-grained token with read access to this one repository. The token is typed into a hidden prompt and is not saved in the notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DATA_DIR, exist_ok=True)
print("Data folder:", DATA_DIR)

In [ ]:
# Option A: upload the ZIP you downloaded from GitHub
from google.colab import files
import zipfile, glob, shutil
up = files.upload()                                  # YOU: pick the zip
zipfile.ZipFile(next(iter(up))).extractall("/content/_zip")
src = glob.glob("/content/_zip/*/site_suitability_starter")[0]
shutil.copytree(src, "/content/code", dirs_exist_ok=True)
print(sorted(os.listdir("/content/code")))

In [ ]:
# Option B: clone with a token (skip if you used option A)
from getpass import getpass
import subprocess, shutil
token = getpass("GitHub token (hidden): ")
subprocess.run(["git", "clone", "--depth", "1",
                f"https://{token}@github.com/{REPO}.git", "/content/repo"], check=True)
subprocess.run(["git", "-C", "/content/repo", "remote", "set-url", "origin",
                f"https://github.com/{REPO}.git"], check=True)     # forget the token
del token
shutil.copytree("/content/repo/site_suitability_starter", "/content/code", dirs_exist_ok=True)

In [ ]:
!pip -q install geopandas earthengine-api
%cd /content/code

## 2. Permits (the label source)

Downloads every Austin permit issued since 2021-01-01. This is a few hundred thousand rows and takes several minutes.

**Check:** the `work_class` and `permit_class_mapped` tables below. `build_labels.py` counts a permit as development when the class is Residential or Commercial and `work_class` contains "New". Decide as a team whether that is the right definition.

In [ ]:
!python pull_permits.py --since 2021-01-01 --out "$DATA_DIR/permits_raw.csv"

In [ ]:
import pandas as pd
permits = pd.read_csv(f"{DATA_DIR}/permits_raw.csv", dtype={"tcad_id": str}, low_memory=False)
print(len(permits), "permits;", permits["issue_date"].min(), "to", permits["issue_date"].max())
display(permits["permit_class_mapped"].value_counts(dropna=False).head(10))
display(permits["work_class"].value_counts(dropna=False).head(20))
print("share with coordinates:", permits["latitude"].notna().mean().round(3))
print("example tcad_id values:", permits["tcad_id"].dropna().head(5).tolist())

## 3. Parcel test pull and the id field

Pulls three pages (3,000 parcels) of **today's** parcels from Travis County. This is only to learn the field names. It is not the 2020 snapshot.

**Check:** which parcel field holds values shaped like the permits' `tcad_id` (10-digit text). The layer metadata suggests `geo_id`. The cell prints the share of test parcels whose id appears in the permits for each candidate.

In [ ]:
!python pull_parcels.py --out "$DATA_DIR/parcels_test.geojson" --max-pages 3

In [ ]:
import geopandas as gpd
test = gpd.read_file(f"{DATA_DIR}/parcels_test.geojson")
print(list(test.columns))
norm = lambda s: s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
permit_ids = set(norm(permits["tcad_id"].dropna()))
for col in test.columns.drop("geometry"):
    share = norm(test[col]).isin(permit_ids).mean()
    if share > 0.01:
        print(f"{col:<20} {share:.1%} of test parcels have an id that appears in the permits;"
              f" examples {test[col].head(3).tolist()}")

## 4. The 2020 parcel snapshot and appraisal roll (YOU: download and upload)

These two files cannot be fetched automatically. Follow `docs/DATA_SOURCES.md`, then put both files in the Drive data folder:

- `parcels_2020.geojson` (or `.gpkg`): parcel polygons dated 2020 or earlier.
- `appraisal_2020.csv`: one row per property with an id, a land value and an improvement value from the 2020 certified roll.

Then fill in the settings cell at the top (`PARCEL_ID_FIELD`, `PARCEL_YEAR`, the appraisal column names) and **run the settings cell again**. The next cell checks them.

In [ ]:
import os
assert PARCEL_YEAR and PARCEL_YEAR <= CUTOFF_YEAR, "Set PARCEL_YEAR (2020 or earlier) in the settings cell."
assert APPRAISAL_YEAR and APPRAISAL_YEAR <= CUTOFF_YEAR, "Set APPRAISAL_YEAR (2020 or earlier)."
for f in (PARCELS_2020, APPRAISAL_CSV):
    assert os.path.exists(f"{DATA_DIR}/{f}"), f"Missing {f} in {DATA_DIR}"
p2020 = gpd.read_file(f"{DATA_DIR}/{PARCELS_2020}", rows=5)
print("parcel columns:", list(p2020.columns))
roll = pd.read_csv(f"{DATA_DIR}/{APPRAISAL_CSV}", dtype=str, nrows=5)
print("appraisal columns:", list(roll.columns))
for name, value, cols in [("PARCEL_ID_FIELD", PARCEL_ID_FIELD, p2020.columns),
                          ("LABELS_JOIN_COL", LABELS_JOIN_COL, p2020.columns),
                          ("APPRAISAL_ID_COL", APPRAISAL_ID_COL, roll.columns),
                          ("LAND_VALUE_COL", LAND_VALUE_COL, roll.columns),
                          ("IMPROVEMENT_VALUE_COL", IMPROVEMENT_VALUE_COL, roll.columns)]:
    assert value in cols, f"{name} = {value!r} is not a column. Fix the settings cell."
print("Settings look consistent.")

## 5. Labels

Matches each qualifying 2021–2025 permit to the 2020 parcel it falls inside, and writes `developed` = 0 or 1 for every parcel.

**Check the printed counts:** how many permits have coordinates, how many landed inside a parcel, and the final developed count. A very low match count means the parcel file and the permits do not line up.

In [ ]:
!python build_labels.py --permits "$DATA_DIR/permits_raw.csv" --parcels "$DATA_DIR/$PARCELS_2020" \
    --parcel-id-field "$PARCEL_ID_FIELD" --label-mode location --cutoff-year $CUTOFF_YEAR \
    --negative-sample-ratio 0 --out "$DATA_DIR/labeled_parcels.geojson"

## 6. Earth Engine: sign in, pilot, then the full run (YOU: sign in)

**Before this step** you need a Google Cloud project registered for Earth Engine (free for education): https://earthengine.google.com/ → *Get Started*. Put its ID in `GCP_PROJECT` in the settings cell.

The sign-in cell opens a Google prompt. The extraction cells use `%run` so the script runs inside this notebook and shares that sign-in.

If `ee.Initialize` fails with a credentials error, replace the first line with `ee.Authenticate(auth_mode="notebook")` and follow the link it prints.

In [ ]:
import ee
assert GCP_PROJECT, "Set GCP_PROJECT in the settings cell."
ee.Authenticate()
ee.Initialize(project=GCP_PROJECT)
print("Earth Engine is ready. Test value:", ee.Number(1).add(1).getInfo())

**Pilot (50 parcels).** Takes a minute or two. Do not start the full run until the checks below look right.

In [ ]:
%run ee_extract_embeddings.py --project {GCP_PROJECT} --parcels "{DATA_DIR}/labeled_parcels.geojson" --out "{DATA_DIR}/embeddings_pilot.csv" --terrain-out "{DATA_DIR}/terrain_pilot.csv" --pilot 50

In [ ]:
pilot = pd.read_csv(f"{DATA_DIR}/embeddings_pilot.csv")
emb_cols = [c for c in pilot.columns if c.startswith("emb_A")]
print("rows:", len(pilot), "| embedding columns:", len(emb_cols))
print("rows with no embedding:", int(pilot["emb_A00"].isna().sum()))
print("embedding value range:", float(pilot[emb_cols].min().min()), "to", float(pilot[emb_cols].max().max()), "(expect within -1..1)")
print("emb_change range:", pilot["emb_change"].min(), "to", pilot["emb_change"].max(), "(expect small, >= 0)")
ter = pd.read_csv(f"{DATA_DIR}/terrain_pilot.csv")
display(ter.describe(include="all").T)       # elevation in meters, slope in percent, land cover names

**Full run.** This can take hours for a whole city. It is safe to stop and rerun: finished parcels are skipped. If Colab disconnects, rerun step 1's Drive cell, the install cell, the sign-in cell, and then this cell.

In [ ]:
%run ee_extract_embeddings.py --project {GCP_PROJECT} --parcels "{DATA_DIR}/labeled_parcels.geojson" --out "{DATA_DIR}/embeddings.csv" --terrain-out "{DATA_DIR}/terrain.csv"

## 7. Other layers (YOU: download the ones that need a browser)

Put these in the Drive data folder. See `docs/DATA_SOURCES.md` for links. Any layer you skip is simply left out of the feature table, and step 8 tells you how to run the model without it.

| File name expected here | What |
|---|---|
| `zoning.geojson` | City of Austin zoning polygons (current zoning: a stated limitation) |
| `flood_hazard.geojson` | FEMA flood hazard area polygons |
| `roads_2020.zip` | Census TIGER/Line 2020 roads for Travis County (the next cell tries to download it) |
| `austin_full_purpose.geojson` | City limits polygon |

In [ ]:
# TIGER/Line 2020 roads for Travis County (FIPS 48453). If this fails, download it in a browser.
!wget -q -nc -O "$DATA_DIR/roads_2020.zip" https://www2.census.gov/geo/tiger/TIGER2020/ROADS/tl_2020_48453_roads.zip || echo "Download failed: get the file by hand."
!ls -la "$DATA_DIR"

## 8. Build the feature table

Adds the tabular columns, keeps only parcels that were vacant or underused in 2020, and joins the embeddings.

**Check the printed lines:** the appraisal match rate (a low rate means a wrong id column), how many parcels pass the vacant/underused filter, and the final developed share.

In [ ]:
layers = {"--zoning": "zoning.geojson", "--floodplain": "flood_hazard.geojson",
          "--roads": "roads_2020.zip", "--city-limits": "austin_full_purpose.geojson"}
extra = []
for flag, name in layers.items():
    path = f"{DATA_DIR}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        extra += [flag, path]
    else:
        print(f"skipping {flag}: {name} not found")
cmd = ["python", "build_features.py",
       "--labels", f"{DATA_DIR}/labeled_parcels.geojson", "--parcel-year", str(PARCEL_YEAR),
       "--appraisal", f"{DATA_DIR}/{APPRAISAL_CSV}", "--appraisal-year", str(APPRAISAL_YEAR),
       "--appraisal-id-col", APPRAISAL_ID_COL, "--labels-join-col", LABELS_JOIN_COL,
       "--land-value-col", LAND_VALUE_COL, "--improvement-value-col", IMPROVEMENT_VALUE_COL,
       "--max-improvement-ratio", str(MAX_IMPROVEMENT_RATIO),
       "--terrain", f"{DATA_DIR}/terrain.csv", "--embeddings", f"{DATA_DIR}/embeddings.csv",
       "--out", f"{DATA_DIR}/features.csv"] + extra
import subprocess
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout, r.stderr)

## 9. Train and compare the three models

If step 8 printed a `--numeric-cols ... --categorical-cols ...` line because a layer was missing, paste those options into `EXTRA_ARGS` below.

`--strict-leakage` is on: the run stops if any single column nearly separates the classes or has a suspicious name. Do not turn it off to get past a warning. Find out why the column leaks.

In [ ]:
EXTRA_ARGS = ""      # e.g. "--numeric-cols lot_area slope elevation improvement_value_ratio --categorical-cols land_cover_class"
!python train_models.py --data "$DATA_DIR/features.csv" --outdir "$DATA_DIR/results" \
    --tune-C --robustness --imagery-breakdown --strict-leakage $EXTRA_ARGS

## 10. Read the results

Start with `comparison.txt`. Report the **spatial** cross-validation numbers. `docs/RESULTS_GUIDE.md` says how to word each possible outcome.

In [ ]:
print(open(f"{DATA_DIR}/results/comparison.txt").read())
from IPython.display import Image, display
display(Image(f"{DATA_DIR}/results/auc_by_model.png", width=800))
display(Image(f"{DATA_DIR}/results/risk_map.png", width=1000))

## Attribution

- "The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind." (CC-BY 4.0)
- Permits: City of Austin Open Data. Parcels and appraisal values: Travis Central Appraisal District. Elevation: USGS 3DEP. Land cover: USGS NLCD 2019.
- AI use: see `AI_ACKNOWLEDGMENT.md`.